# Tuning chung cho báo cáo

Đọc một CSV đã làm sạch, dùng pipeline mới từ `src/model.py` và `src/tuning.py`. Các thí nghiệm lịch sử dùng `models_exp.py` và `tuning_exp.py`.

Luồng: CSV theo giờ → chỉ lấy train → tạo feature → cùng fold và ngân sách → chọn theo MAE → một JSON kết quả. Notebook này không đánh giá validation/test. Chưa chạy tìm kiếm đầy đủ; chọn Run All khi muốn huấn luyện.

## 1. Nạp module

Tìm gốc dự án theo CSV tiền xử lý. Khởi động lại kernel nếu vừa đổi tên module để tránh dùng bản cũ trong bộ nhớ.

In [1]:
from pathlib import Path
import sys
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'data/preprocessed/hourly_unfilled.csv').is_file())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
import pandas as pd
from IPython.display import Markdown, display
from src.model import MODEL_NAMES
from src.tuning import load_training_data, describe_folds, run_tuning, results_table, candidate_parameters

## 2. Cấu hình dữ liệu

A12/B27 là đối chứng chính trên toàn bộ giờ train. Nếu thêm C30 vào CONFIGS, cả lượt sẽ dùng cùng cohort đủ ba history; cần trình bày riêng với lượt A/B đầy đủ. Hàm chỉ đọc target của 70% train, không tải target validation/test.

In [2]:
CONFIGS = ('A12', 'B27')
N_SPLITS = 3
N_CANDIDATES = 6
data = load_training_data(project_root / 'data/preprocessed/hourly_unfilled.csv', configs=CONFIGS)
display(pd.DataFrame([data.metadata]).rename(columns={'n_cohort': 'n'}))

,input_path,input_sha256,n_total,n_raw_train,n,cohort,train_end,scope,weather,history
0,E:\Project\SGU_HocMay_2026-2027\metro_intersta...,d2092b0060b8db304a5da8c27b637b38da97b798e43f67...,40575,28402,28402,all observed train hours,2017-05-10 04:00:00,train CV only; validation/test targets not loaded,same-hour weather assumed available,sequential observed past only; mean >=18/24 hours


## 3. Fold và ngân sách

Dùng đúng các fold cho mọi mô hình/cấu hình. Cùng số ứng viên không đồng nghĩa cùng thời gian chạy. Ridge/MLP chuẩn hóa trong fold; lịch giữ đúng danh sách A12/B27, không dùng one-hot cặp thứ–giờ của nhánh cũ. Mỗi pipeline học median từ đúng fold-train và chỉ tra nhiệt độ gốc trong quá khứ.

In [3]:
folds = describe_folds(data, N_SPLITS)
display(folds)
budget = pd.DataFrame([dict(model=name, candidates=len(candidate_parameters(name, N_CANDIDATES)),
                           folds=N_SPLITS, configurations=len(CONFIGS),
                           fits=N_CANDIDATES*N_SPLITS*len(CONFIGS), n=len(data.frame))
                       for name in MODEL_NAMES])
display(budget)

,fold,n_fit,n,fit_start,fit_end,fit_calendar_hours,fit_sha256,score_start,score_end,score_calendar_hours,score_sha256
0,1,7102,7100,2012-10-02 09:00:00,2013-08-18 18:00:00,7690.0,5647db9e5c95a8fa1433eea2fb454f9840494e586322c5...,2013-08-18 19:00:00,2015-07-07 11:00:00,16505.0,ca6ac42b741fd263c3840d6283d9ab9620a2e89b9dd56e...
1,2,14202,7100,2012-10-02 09:00:00,2015-07-07 11:00:00,24195.0,bc96aafb81458110fc481a07a145544309d64a0654a60d...,2015-07-07 12:00:00,2016-07-15 04:00:00,8969.0,ef63ea2383b369ac60f4985a15431d5b26cb67ed821856...
2,3,21302,7100,2012-10-02 09:00:00,2016-07-15 04:00:00,33164.0,857bd760277e06d2942537fafae2e7de42296a123976d3...,2016-07-15 05:00:00,2017-05-10 04:00:00,7176.0,ee777d7edc5c513a1e155e4c4f40b782779ec8356cab25...


,model,candidates,folds,configurations,fits,n
0,Ridge,6,3,2,36,28402
1,Random Forest,6,3,2,36,28402
2,Gradient Boosting,6,3,2,36,28402
3,MLP,6,3,2,36,28402


## 4. Tìm tham số trên train

Ô này thực sự huấn luyện các ứng viên. Không dùng early stopping dựa trên validation ngoài. Chọn MAE CV trung bình thấp nhất, RMSE phá hòa; ghi thêm R², số mẫu, khoảng lịch và cảnh báo hội tụ. Lưu một tuning_results.json, gồm các ứng viên và dự đoán ngoài fold của ứng viên tốt nhất; không nhân bản CSV đầu vào.

In [4]:
output_path = project_root / 'notebooks/tuning/outputs/tuning_results.json'
result = run_tuning(data, n_candidates=N_CANDIDATES, n_splits=N_SPLITS, output_path=output_path,
                    progress_callback=lambda step: print(step, flush=True))

{'config': 'A12', 'model': 'Ridge', 'candidate': 1, 'fold': 1, 'MAE': 569.4799011746259}
{'config': 'A12', 'model': 'Ridge', 'candidate': 1, 'fold': 2, 'MAE': 574.614130584287}
{'config': 'A12', 'model': 'Ridge', 'candidate': 1, 'fold': 3, 'MAE': 578.4068934824769}
{'config': 'A12', 'model': 'Ridge', 'candidate': 2, 'fold': 1, 'MAE': 569.479999879649}
{'config': 'A12', 'model': 'Ridge', 'candidate': 2, 'fold': 2, 'MAE': 574.6142024510343}
{'config': 'A12', 'model': 'Ridge', 'candidate': 2, 'fold': 3, 'MAE': 578.4068877744003}
{'config': 'A12', 'model': 'Ridge', 'candidate': 3, 'fold': 1, 'MAE': 569.4811253851233}
{'config': 'A12', 'model': 'Ridge', 'candidate': 3, 'fold': 2, 'MAE': 574.6148688517826}
{'config': 'A12', 'model': 'Ridge', 'candidate': 3, 'fold': 3, 'MAE': 578.4070547012544}
{'config': 'A12', 'model': 'Ridge', 'candidate': 4, 'fold': 1, 'MAE': 569.4924518080161}
{'config': 'A12', 'model': 'Ridge', 'candidate': 4, 'fold': 2, 'MAE': 574.6218682259573}
{'config': 'A12', 'mode

## 5. Tổng kết để bàn giao

Bảng dưới chỉ là CV dùng chọn tham số, chưa phải kết quả validation/test. Ensemble đọc best_params và dự đoán ngoài fold từ JSON này; không tìm lại tham số mô hình thành phần. fit_selected_model chỉ fit lại từ tham số đã chọn, dùng khi bước validation được triển khai.

In [5]:
display(results_table(result))
display(Markdown(f'Kết quả được lưu trong `{output_path.relative_to(project_root).as_posix()}`. Chưa đánh giá validation hoặc test.'))

,config,model,best_params,MAE_CV,RMSE_CV,n
0,A12,Ridge,{'model__alpha': 0.001},574.166975,743.546006,21300
1,A12,Random Forest,"{'model__n_estimators': 200, 'model__min_sampl...",280.512758,461.300395,21300
2,A12,Gradient Boosting,"{'model__n_estimators': 200, 'model__max_depth...",295.655647,466.406633,21300
3,A12,MLP,{'model__regressor__learning_rate_init': 0.003...,292.679958,467.455191,21300
4,B27,Ridge,{'model__alpha': 0.001},572.908291,740.696150,21300
5,B27,Random Forest,"{'model__n_estimators': 400, 'model__min_sampl...",281.194339,467.137537,21300
6,B27,Gradient Boosting,"{'model__n_estimators': 200, 'model__max_depth...",305.154243,474.113970,21300
7,B27,MLP,{'model__regressor__learning_rate_init': 0.001...,337.513329,516.842898,21300


Kết quả được lưu trong `notebooks/tuning/outputs/tuning_results.json`. Chưa đánh giá validation hoặc test.